# Hello World - Jupyter Notebook Demo

Simple chat completion using OGX from a Jupyter notebook.

In [1]:
import os
import requests
from pathlib import Path
from openai import OpenAI

import sys                                                                                                                                                                                                        
print(sys.version)                      
print(sys.executable)      

3.12.12 (main, Jan 16 2026, 00:00:00) [GCC 15.2.1 20251211 (Red Hat 15.2.1-5)]
/home/dhiggins/workarea/ls_claude/lls-showroom/llama-stack-showroom/.venv/bin/python3


In [ ]:
# Load configuration from environment variables or K8s cluster
import subprocess

def get_route_url(name, namespace="redhat-ods-applications"):
    try:
        host = subprocess.run(
            ["oc", "get", "route", name, "-n", namespace, "-o", "jsonpath={.spec.host}"],
            capture_output=True, text=True, timeout=10
        ).stdout.strip()
        if not host:
            return None
        tls = subprocess.run(
            ["oc", "get", "route", name, "-n", namespace, "-o", "jsonpath={.spec.tls}"],
            capture_output=True, text=True, timeout=10
        ).stdout.strip()
        return f"{'https' if tls else 'http'}://{host}"
    except Exception:
        return None

def get_secret(secret_name, key, namespace="redhat-ods-applications"):
    try:
        import base64
        encoded = subprocess.run(
            ["oc", "get", "secret", secret_name, "-n", namespace, "-o", f"jsonpath={{.data.{key}}}"],
            capture_output=True, text=True, timeout=10
        ).stdout.strip()
        return base64.b64decode(encoded).decode() if encoded else None
    except Exception:
        return None

ogx_url = os.environ.get('OGX_URL') or get_route_url("ogx-distribution")
keycloak_url = os.environ.get('KEYCLOAK_URL') or get_route_url("keycloak")
username = os.environ.get('KEYCLOAK_USERNAME') or "admin"
password = os.environ.get('KEYCLOAK_ADMIN_PASSWORD') or get_secret("keycloak-secret", "KEYCLOAK_ADMIN_PASSWORD")
client_secret = os.environ.get('KEYCLOAK_CLIENT_SECRET') or get_secret("keycloak-secret", "KEYCLOAK_CLIENT_SECRET")

print(f"Connecting to: {ogx_url}")

In [ ]:
# Get authentication token if Keycloak is configured
api_key = "not-needed"

if keycloak_url and username and password and client_secret:
    print(f"\nAuthenticating with Keycloak as '{username}'...")
    
    token_url = f"{keycloak_url}/realms/ogx-demo/protocol/openid-connect/token"
    payload = {
        'client_id': 'ogx',
        'client_secret': client_secret,
        'username': username,
        'password': password,
        'grant_type': 'password'
    }
    
    response = requests.post(token_url, data=payload, verify=True)
    response.raise_for_status()
    
    token_data = response.json()
    api_key = token_data.get('access_token')
    
    print(f"Authentication successful")
    print(f"  Token type: {token_data.get('token_type', 'Bearer')}")
    print(f"  Expires in: {token_data.get('expires_in', 'unknown')} seconds")

# Initialize OpenAI client
client = OpenAI(
    base_url=f"{ogx_url}/v1",
    api_key=api_key,
)

In [ ]:
# Send a simple chat completion request
print("\nSending chat completion request...")
print("Prompt: 'Say hello from a Jupyter notebook!'")

response = client.chat.completions.create(
    model="vllm-inference/llama-3-2-3b",
    messages=[
        {"role": "user", "content": "Say hello from a Jupyter notebook!"}
    ],
    max_tokens=100
)

print("\nResponse:", response.choices[0].message.content)

In [ ]:
# Verify response is not empty
assert response.choices[0].message.content, "Expected non-empty response"
print("\n✓ Notebook demo completed successfully")
print(f"  Model: {response.model}")
print(f"  Tokens: {response.usage.total_tokens} total")